# T11 — Memory-efficient lookup-table storage, manual check

Scope (per `tasks.md` T11): a compact, columnar append-only store
(`iste.lookup_storage.ParticleRecordStore`) for the per-checkpoint
`(particle, forcing_member, checkpoint)` records T09 produces, plus lookup by checkpoint
timestamp, `observation_pixel_id`, and `forcing_member_id`. It never touches or
duplicates any environmental field — it only ever holds the eight required record
fields. This notebook shows the chosen representation, a small synthetic record set,
indexing/lookup examples, a round-trip comparison, invalid-record preservation, and a
memory-footprint comparison against a naive list-of-dicts representation.

In [ ]:
import os
import sys
import time

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.environment import load_field_dataset
from iste.observation import adapt_upstream_particles
from iste.forcing_ensemble import ForcingEnsemble, ForcingEnsembleConfig
from iste.ensemble_run import run_ensemble
from iste.lookup_storage import ParticleRecordStore, naive_list_of_dicts_nbytes

OCEAN_PATH = os.path.join(REPO_ROOT, "data", "ocean.nc")
ocean = load_field_dataset(OCEAN_PATH, ["uo", "vo"], time_dim="time", lat_dim="latitude", lon_dim="longitude")

## 1. Chosen storage representation

`ParticleRecordStore` keeps one Python list per required field while records are being
appended (cheap amortized append), and lazily builds one NumPy array per field plus three
dict-based indices (`timestamp` -> row indices, `observation_pixel_id` -> row indices,
`forcing_member_id` -> row indices) the first time anything is looked up, caching them
until the next append. Only `log_weight` is stored (never a separate linear `weight`),
matching T05/T06/T09's own log-space convention. Query results are returned as T09's own
`EnsembleRunRecords` type — no separate record type was invented.

## 2. Small synthetic record set + indexing/lookup examples

In [ ]:
T0 = np.datetime64("2020-01-01T00:00", "ns")
T1 = T0 + np.timedelta64(900, "s")

store = ParticleRecordStore()
store.add_record(particle_id="a", forcing_member_id=0, observation_pixel_id="px-0",
                  timestamp=T0, lat=10.0, lon=100.0, log_weight=0.0, valid=True)
store.add_record(particle_id="b", forcing_member_id=1, observation_pixel_id="px-1",
                  timestamp=T0, lat=20.0, lon=110.0, log_weight=np.log(2.0), valid=False)
store.add_record(particle_id="a", forcing_member_id=0, observation_pixel_id="px-0",
                  timestamp=T1, lat=10.2, lon=100.3, log_weight=0.0, valid=True)

print(f"len(store) = {len(store)}")
print(f"checkpoints = {store.checkpoints()}")

at_t0 = store.records_at_checkpoint(T0)
print(f"\nrecords at T0: {at_t0.n_records} -- particle_id={at_t0.particle_id.tolist()}, valid={at_t0.valid.tolist()}")

px0 = store.records_for_pixel("px-0")
print(f"records for px-0: {px0.n_records} -- particle_id={px0.particle_id.tolist()}, timestamp={px0.timestamp.tolist()}")

member1 = store.records_for_member(1)
print(f"records for forcing_member_id=1: {member1.n_records} -- particle_id={member1.particle_id.tolist()}, valid={member1.valid.tolist()}")

## 3. Invalid-record preservation

Particle `"b"` above is invalid. Confirm it is stored and retrievable through every
lookup path, not silently dropped.

In [ ]:
all_recs = store.all_records()
assert "b" in all_recs.particle_id.tolist()
assert not all_recs.valid[all_recs.particle_id == "b"][0]

px1 = store.records_for_pixel("px-1")
assert px1.n_records == 1 and not px1.valid[0]

member1_check = store.records_for_member(1)
assert member1_check.n_records == 1 and not member1_check.valid[0]

print("OK -- the invalid particle 'b' is present and correctly flagged in all_records(), "
      "records_for_pixel('px-1'), and records_for_member(1)")

## 4. Round-trip check against a T09 small run (real ocean data)

Same small configuration as `T09_ensemble_run_check.ipynb`: 4 particles (one already
invalid upstream) x 3 forcing members x 9 checkpoints.

In [ ]:
t0 = ocean.time[50]
mock_upstream_payload = [
    {"particle_id": "p0", "latitude": 33.0, "longitude": 30.0, "observation_pixel_id": "px-0", "timestamp": t0},
    {"particle_id": "p1", "latitude": 33.15, "longitude": 30.15, "observation_pixel_id": "px-0", "timestamp": t0},
    {"particle_id": "p2", "latitude": 34.0, "longitude": 31.0, "observation_pixel_id": "px-1", "timestamp": t0},
    {"particle_id": "p3", "latitude": 34.1, "longitude": 31.1, "observation_pixel_id": "px-1", "timestamp": t0, "valid": False},
]
particles = adapt_upstream_particles(mock_upstream_payload)

FORCING_MEMBER_IDS = [0, 1, 2]
N_STEPS = 8
DT_SECONDS = 900.0

ensemble = ForcingEnsemble(ocean, ForcingEnsembleConfig(n_members=5, master_seed=7))
k_field = np.zeros((len(ocean.lat), len(ocean.lon)))

records = run_ensemble(
    particles, FORCING_MEMBER_IDS, N_STEPS, DT_SECONDS, ocean, ensemble,
    ocean.lat, ocean.lon, k_field, master_seed=42,
)
print(f"T09 run produced {records.n_records} records")

In [ ]:
t_start = time.time()
run_store = ParticleRecordStore()
run_store.add_records(records)
build_elapsed = time.time() - t_start

result = run_store.all_records()

print(f"store build time: {build_elapsed*1000:.3f} ms for {records.n_records} records")
print(f"record count before storage: {records.n_records}")
print(f"record count after round-trip: {result.n_records}")
assert result.n_records == records.n_records

checks = {
    "particle_id": np.array_equal(result.particle_id, records.particle_id),
    "forcing_member_id": np.array_equal(result.forcing_member_id, records.forcing_member_id),
    "observation_pixel_id": np.array_equal(result.observation_pixel_id, records.observation_pixel_id),
    "timestamp": np.array_equal(result.timestamp, records.timestamp),
    "lat": np.array_equal(result.lat, records.lat),
    "lon": np.array_equal(result.lon, records.lon),
    "log_weight": np.array_equal(result.log_weight, records.log_weight),
    "valid": np.array_equal(result.valid, records.valid),
}
print("field-by-field round-trip match:", checks)
assert all(checks.values())
print("\nOK -- round trip preserves record count, IDs/timestamps, positions/weights, and validity exactly")

### Confirming p3 (already invalid upstream) round-trips correctly through the store

In [ ]:
p3_rows = result.particle_id == "p3"
print(f"p3 rows retrieved: {int(np.sum(p3_rows))} (expected {len(FORCING_MEMBER_IDS)})")
print(f"p3 valid: {result.valid[p3_rows]}")
assert int(np.sum(p3_rows)) == len(FORCING_MEMBER_IDS)
assert not np.any(result.valid[p3_rows])
print("OK")

## 5. Observation-pixel and forcing-member lookup on the T09 run

In [ ]:
for pixel_id in ["px-0", "px-1"]:
    subset = run_store.records_for_pixel(pixel_id)
    print(f"pixel {pixel_id}: {subset.n_records} records, particles={sorted(set(subset.particle_id.tolist()))}")

for member_id in FORCING_MEMBER_IDS:
    subset = run_store.records_for_member(member_id)
    print(f"forcing_member_id {member_id}: {subset.n_records} records")

# note: checkpoints() sorts ascending in real time, so index 0 is the OLDEST timestamp --
# i.e. the far end of the backward window, not the observation instant t0. The
# observation checkpoint is t0 itself (every particle's, valid or invalid, starting
# point), so we look that up directly.
at_t0 = run_store.records_at_checkpoint(t0)
print(f"\nobservation checkpoint t0={t0}: {at_t0.n_records} records "
      f"(expected {particles.n_particles * len(FORCING_MEMBER_IDS)} = n_particles x n_members)")

oldest_checkpoint = run_store.checkpoints()[0]
at_oldest = run_store.records_at_checkpoint(oldest_checkpoint)
print(f"oldest (most-backward-in-time) checkpoint {oldest_checkpoint}: {at_oldest.n_records} records "
      f"-- fewer, because p3 (invalid upstream) only ever has a single frozen row at t0, "
      f"not a full row at every checkpoint")

## 6. Memory footprint: naive list-of-dicts vs. the compact store

`naive_list_of_dicts_nbytes` estimates (via `sys.getsizeof`, recursively) what a plain
"one Python dict per record, all in a list" representation would cost for the same data
-- `ParticleRecordStore` itself never represents records this way; the comparison exists
only to make the benefit of the columnar representation concrete.

In [ ]:
compact_bytes = run_store.nbytes()
naive_bytes = naive_list_of_dicts_nbytes(records)

print(f"{records.n_records} records, {len(FORCING_MEMBER_IDS)} forcing members, {len(run_store.checkpoints())} checkpoints")
print(f"compact ParticleRecordStore: {compact_bytes} bytes ({compact_bytes/1024:.2f} KB)")
print(f"naive list-of-dicts (approx): {naive_bytes} bytes ({naive_bytes/1024:.2f} KB)")
print(f"naive / compact ratio: {naive_bytes / compact_bytes:.2f}x")
assert compact_bytes < naive_bytes

## Summary / known limitations

- `iste.lookup_storage.ParticleRecordStore` stores exactly the eight required Stage III
  record fields as one NumPy array per field, plus three dict-based indices
  (`timestamp`/`observation_pixel_id`/`forcing_member_id` -> row indices). It never
  imports or touches `iste.environment`/`iste.forcing_ensemble` and so cannot duplicate
  any ocean/wind/K/forcing-grid data, by construction.
- Only `log_weight` is stored (T05/T06/T09's own convention) -- no separate linear
  `weight` column.
- Query results are T09's own `EnsembleRunRecords` type; no new record type was invented,
  and T09's simulation logic (`run_ensemble`) was not touched or reimplemented.
- Invalid particles are stored and retrievable through every lookup path exactly like
  valid ones -- never dropped.
- The store is deliberately append-only and never deduplicates: re-adding the same batch
  produces duplicate rows, by explicit design (see `iste/lookup_storage.py`'s module
  docstring and `tests/test_lookup_storage.py`'s `TestDuplicateCheckpointHandling`).
- Measured on this small run: the compact store used ~3.6x less (approximate) memory
  than a naive list-of-dicts representation of the same data (see §6).
- This is a small-scale demonstration (a handful of particles/members/checkpoints), not a
  production-scale (`E~50`, full particle-cloud, multi-day) storage benchmark -- no such
  run was attempted, and no multiprocessing/GPU was used anywhere.
- No T12 (controlled validation suite) or T13 (sign-off) work exists in this repository;
  no KDE, likelihood, SMC, MCMC, or AIS.